In [6]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
from sklearn.svm import SVC, LinearSVC
import re 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN
import matplotlib.pyplot as plt

# definizioni

In [7]:
df = pd.read_csv("winter_project_2026/development.csv")

category_map = {
    0: "International News",
    1: "Business",
    2: "Technology",
    3: "Entertainment",
    4: "Sports",
    5: "General News",
    6: "Health"
}

df.dtypes

#remove nan

df.dropna(inplace=True)
df.isna().sum().sum()

np.int64(0)

In [ ]:
def clean_text__final(text):
    if not isinstance(text, str):
        return ""

    urls = re.findall(r'https?://\S+', text)
    url_keywords = []
    
    web_junk = {'jpg', 'jpeg', 'png', 'gif', 'html', 'php', 'sig', 'dc', 'nm', 'rss', 'http', 'https', 'www', 'img'}

    for url in urls:
        parts = re.split(r'[-/;._, \s]+', url)
        
        words = []
        for p in parts:
            words.append(re.sub(r"[^a-zA-Z']", '', p))
        
        for w in words:
            if w not in web_junk and 3 < len(w) < 15:
                url_keywords.append(w)

    url_info = " ".join(url_keywords)

    text_without_tags = re.sub(r'<[^>]+>', ' ', text)

    captions = re.findall(r'alt="([^"]*)"', text)
    captions = " ".join(captions)
    captions = re.sub(r'[^a-zA-Z\s]', ' ', captions)

    text = re.sub(r"[^a-zA-Z\s']", ' ', text_without_tags)
    # 4. Combine Article + Extracted URL Info
    combined = text + " " + url_info + " " + captions
    
    return re.sub(r'\s+', ' ', combined).strip().lower()

def clean_text_without_url(text):
    if not isinstance(text, str):
        return ""

    # capturing alt captions from HTML before stripping tags
    captions = " ".join(re.findall(r'alt="([^"]*)"', text))
    captions = re.sub(r'[^a-zA-Z\s]', ' ', captions)

    # removing the URL messages 
    text_no_links = re.sub(r'https?://\S+', ' ', text)

    # remove HTML tags
    text_without_tags = re.sub(r'<[^>]+>', ' ', text_no_links)

    # keep information about money simbols
    text_without_tags = re.sub(
        r'[$€£¥]',
        ' MONEYAMOUNT ',
        text_without_tags
    )

    # this removes punctuation, numbers
    text_final = re.sub(r"[^a-zA-Z\s]", ' ', text_without_tags)

    combined = text_final  + " " + captions

    # 6. lower casse and extra whitespace removal
    return re.sub(r'\s+', ' ', combined).strip().lower()

clean_text = clean_text_without_url


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer 
import nltk
from nltk.tokenize import word_tokenize 
from nltk.stem.wordnet import WordNetLemmatizer 
from nltk.corpus import stopwords as sw 

# nltk.download('punkt')
# nltk.download('wordnet')
# nltk.download('stopwords')

class LemmaTokenizer1(object): 
    def __init__(self): 
        self.lemmatizer = WordNetLemmatizer() 
        self.stop_words = set(sw.words('english'))
    
    def __call__(self, document): 
        lemmas = [] 
        for t in word_tokenize(document): 
            t = t.lower().strip()
            
            if t not in self.stop_words:
                lemma = self.lemmatizer.lemmatize(t) 
                lemmas.append(lemma)
                
        return lemmas
    
class LemmaTokenizerGood(object): 
    def __init__(self): 
        self.lemmatizer = WordNetLemmatizer() 
        self.stop_words = set(sw.words('english'))
    
    def __call__(self, document): 
        lemmas = [] 
        for t in word_tokenize(document): 
            t = t.lower().strip()
            
            if t.isalpha() and t not in self.stop_words and len(t) > 2:
                lemma = self.lemmatizer.lemmatize(t) 
                lemmas.append(lemma)
                
        return lemmas

lemmaTokenizer = LemmaTokenizerGood()   
text_transformer1 = TfidfVectorizer(
    max_features=1000000, 
    tokenizer=lemmaTokenizer, 
    stop_words=sw.words('english'), 
    ngram_range=(1, 3), 
    sublinear_tf=True, 
    norm='l2', 
    use_idf = False, 
    token_pattern=None, 
    min_df=3, 
    max_df=0.95)  

text_transformer_classic = TfidfVectorizer(
    max_features=None,
    stop_words='english', 
    ngram_range=(1,3), 
    sublinear_tf=True, 
    norm='l2', 
    use_idf = True, 
    # token_pattern=None, 
    min_df=5, 
    max_df=0.9) 

# test su clean text


In [ ]:
test_sentence = "The articles was being published on <https://news.com/tech-data-2024>. It's houses and cats!"
lemmaTokenizer = LemmaTokenizer1()
print(f"--- FRASE INIZIALE ---\n{test_sentence}\n")
cleaned = clean_text(test_sentence)
print(f"--- DOPO CLEAN_TEXT ---\n{cleaned}\n")

tokens = lemmaTokenizer(cleaned)
print(f"--- DOPO LEMMATOKENIZER ---\n{tokens}")

--- FRASE INIZIALE ---
The articles was being published on <https://news.com/tech-data-2024>. It's houses and cats!

--- DOPO CLEAN_TEXT ---
the articles was being published on it s houses and cats

--- DOPO LEMMATOKENIZER ---
['article', 'published', 'house', 'cat']


# test su lemmaTokenizer

In [ ]:
test_sentence = df.iloc[4]['article']
lemmaTokenizer = LemmaTokenizer1()
print(f"--- FRASE INIZIALE ---\n{test_sentence}\n")
cleaned = clean_text_without_url(test_sentence)
print(f"--- DOPO CLEAN_TEXT ---\n{cleaned}\n")

tokens = lemmaTokenizer(cleaned)
print(f"--- DOPO LEMMATOKENIZER ---\n{tokens}")

--- FRASE INIZIALE ---
<p><a href="http://us.rd.yahoo.com/dailynews/rss/europe/*http://news.yahoo.com/s/nm/20060922/wl_nm/transport_germany_accident_dc"><img src="http://d.yimg.com/us.yimg.com/p/rids/20060922/i/r83709231.jpg?x=47&y=130&sig=dF38yH9MhxP9zBLqrj3SMw--" align="left" height="130" width="47" alt="An aerial view shows the crash site of a Transrapid high-speed train in the northern German region of Emsland near the town of Lathen, September 22, 2006. REUTERS/Bodo Marks" border="0" /></a>Reuters - A high tech German train \
carrying 29 people smashed into a maintenance vehicle on a \
trial run in northern Germany on Friday, killing at least one \
and trapping 10 more in the wreckage, a local official said.</p><br clear="all"/>

--- DOPO CLEAN_TEXT ---
reuters a high tech german train carrying people smashed into a maintenance vehicle on a trial run in northern germany on friday killing at least one and trapping more in the wreckage a local official said an aerial view shows the 

In [ ]:


test = df.iloc[4]['article']
test = test.lower()
urls = re.findall(r'https?://\S+', test)


print(urls)

url_keywords = []
    
web_junk = {'jpg', 'jpeg', 'png', 'gif', 'html', 'php', 'sig', 'dc', 'nm', 'rss', 'http', 'https', 'www', 'img'}

for url in urls:
    parts = re.split(r'[-/;._, \s]+', url)
    print("url parts\n",parts)
    
    words = []
    for p in parts:
        words.append(re.sub(r'[^a-zA-Z]', '', p))
    print("words\n",words)
    
    for w in words:
        if w not in web_junk and 3 < len(w) < 15:
            url_keywords.append(w)

url_info = " ".join(url_keywords)


print("final url info-->\n",url_info)


['http://us.rd.yahoo.com/dailynews/rss/europe/*http://news.yahoo.com/s/nm/20060922/wl_nm/transport_germany_accident_dc"><img', 'http://d.yimg.com/us.yimg.com/p/rids/20060922/i/r83709231.jpg?x=47&y=130&sig=df38yh9mhxp9zblqrj3smw--"']
url parts
 ['http:', 'us', 'rd', 'yahoo', 'com', 'dailynews', 'rss', 'europe', '*http:', 'news', 'yahoo', 'com', 's', 'nm', '20060922', 'wl', 'nm', 'transport', 'germany', 'accident', 'dc"><img']
words
 ['http', 'us', 'rd', 'yahoo', 'com', 'dailynews', 'rss', 'europe', 'http', 'news', 'yahoo', 'com', 's', 'nm', '', 'wl', 'nm', 'transport', 'germany', 'accident', 'dcimg']
url parts
 ['http:', 'd', 'yimg', 'com', 'us', 'yimg', 'com', 'p', 'rids', '20060922', 'i', 'r83709231', 'jpg?x=47&y=130&sig=df38yh9mhxp9zblqrj3smw', '"']
words
 ['http', 'd', 'yimg', 'com', 'us', 'yimg', 'com', 'p', 'rids', '', 'i', 'r', 'jpgxysigdfyhmhxpzblqrjsmw', '']
final url info-->
 yahoo dailynews europe news yahoo transport germany accident dcimg yimg yimg rids


# test sull'estrazione da parte del transformer

In [13]:
lemmaTokenizer = LemmaTokenizerGood()   
text_transformer1 = TfidfVectorizer(
    max_features=1000000, 
    tokenizer=lemmaTokenizer, 
    stop_words=sw.words('english'), 
    ngram_range=(1, 1), 
    sublinear_tf=True, 
    norm='l2', 
    use_idf = False, 
    token_pattern=None, 
)  

text_transformer_classic = TfidfVectorizer(
    max_features=None,
    stop_words='english', 
    ngram_range=(1,1), 
    sublinear_tf=True, 
    norm='l2', 
    use_idf = True, 
    # token_pattern=None, 
) 

In [ ]:
test_sentence = df.iloc[4]['article'] 

print(f"--- ANALISI TF-IDF FRASE 4 ---\n")
print(f"--- frase originale : {test_sentence} \n")

cleaned = True
if cleaned:
    test_sentence = clean_text_without_url(test_sentence)
    print(f"--- frase pulita : {clean_text_without_url(test_sentence)} \n")

text_transformer = text_transformer1 # o'text_transformer1'
tfidf_matrix = text_transformer.fit_transform([test_sentence])

feature_names = text_transformer.get_feature_names_out() 
tfidf_values = tfidf_matrix.toarray().flatten()

test_results = pd.DataFrame({
    'parola': feature_names,
    'tfidf_score': tfidf_values
}).sort_values(by='tfidf_score', ascending=False)


print(test_results)
print(f"\nvocabulary: {text_transformer.vocabulary_}")


--- ANALISI TF-IDF FRASE 4 ---

--- frase originale : <p><a href="http://us.rd.yahoo.com/dailynews/rss/europe/*http://news.yahoo.com/s/nm/20060922/wl_nm/transport_germany_accident_dc"><img src="http://d.yimg.com/us.yimg.com/p/rids/20060922/i/r83709231.jpg?x=47&y=130&sig=dF38yH9MhxP9zBLqrj3SMw--" align="left" height="130" width="47" alt="An aerial view shows the crash site of a Transrapid high-speed train in the northern German region of Emsland near the town of Lathen, September 22, 2006. REUTERS/Bodo Marks" border="0" /></a>Reuters - A high tech German train \
carrying 29 people smashed into a maintenance vehicle on a \
trial run in northern Germany on Friday, killing at least one \
and trapping 10 more in the wreckage, a local official said.</p><br clear="all"/> 

--- frase pulita : reuters a high tech german train carrying people smashed into a maintenance vehicle on a trial run in northern germany on friday killing at least one and trapping more in the wreckage a local official sai

C:\Users\aless\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\feature_extraction\text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['could', 'might', 'must', 'need', 'sha', 'would'] not in stop_words.
  warnings.warn(


In [ ]:
# ------------------
# cleaning prototypes
# -------------------


def remove_slash(text):
    if not isinstance(text, str):
        return ""

    text = text.replace('"', ' ')

    text = text.replace('/', ' ')

    return text
    
def clean_text_emanueliana(text):
    if not isinstance(text, str):
        return ""

    text = re.sub(r'[<>]', ' ', text)
    text = re.sub(r'\b[a-zA-Z_:]+=', ' ', text)
    text = text.replace('"', ' ')
    text = text.replace('/', ' ')
    text = re.sub(r'\s+', ' ', text).strip()

    return text

import re

def clean_text_emanueliana_v2(text):
    if not isinstance(text, str):
        return ""

    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http[s]?://\S+', ' ', text)

    text = text.replace('/', ' ')
    text = text.replace('"', ' ')
    
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text.lower() 

import re

import re

def clean_text_emanueliana_final(text):
    if not isinstance(text, str):
        return ""

    urls = re.findall(r'https?://\S+', text)
    url_keywords = []
    
    web_set = {'jpg', 'jpeg', 'png', 'gif', 'html', 'php', 'sig', 'dc', 'nm', 'rss', 'http', 'https', 'www', 'img'}

    for url in urls:
        parts = re.split(r'[-/;._, \s]+', url)
        
        words = []
        for p in parts:
            words.append(re.sub(r'[^a-zA-Z]', '', p))
        #print("words",words)
        
        for w in words:
            if w not in web_set and 3 < len(w) < 15:
                url_keywords.append(w)

    url_info = " ".join(url_keywords)

    text_without_tags = re.sub(r'<[^>]+>', ' ', text)

    captions = re.findall(r'alt="([^"]*)"', text)
    captions = " ".join(captions)
    captions = re.sub(r'[^a-zA-Z\s]', ' ', captions)

    text = re.sub(r'[^a-zA-Z\s]', ' ', text_without_tags)

    combined = text + " " + url_info + " " + captions

    return re.sub(r'\s+', ' ', combined).strip().lower()

In [16]:
def clean_text(text):
    text = clean_text_emanueliana_final(text)
    return text

In [ ]:

# Filtro lunghezza 
mask_len = df['article'].str.len() >= 50
df = df[mask_len].copy() 
print(f"len -> {(~mask_len).sum()} has been removed")

df = df.reset_index(drop=True)

# rimozioni duplicati
mask_dup = ~df["article"].duplicated()
df = df[mask_dup].reset_index(drop=True)
print(f"len -> {(~mask_dup).sum()} has been removed")


    
def gemini_cleaner_applyer(df):
    df['title'] = df['title'].fillna('').apply(clean_text)
    df['article'] = df['article'].fillna('').apply(clean_text)
    df['source'] = df['source'].fillna('')
    df['combined_text'] = df['title'] + " " + df['article']
    df['combined_text2'] = df['title'] + " " + df['article']

cleaner_applier = gemini_cleaner_applyer




len -> 2729 has been removed
len -> 3580 has been removed


In [18]:
test = df.iloc[4]['article']
test = test.lower()
print("starting text->\n", test)
captions = re.findall(r'alt="([^"]*)"', test)
captions = " ".join(captions)
captions = re.sub(r'[^a-zA-Z\s]', ' ', captions)
print("alt captions->\n", captions)
print("applying removing url function\n", re.sub(r'<[^>]+>', ' ', test))

starting text->
 <p><a href="http://us.rd.yahoo.com/dailynews/rss/europe/*http://news.yahoo.com/s/nm/20060922/wl_nm/transport_germany_accident_dc"><img src="http://d.yimg.com/us.yimg.com/p/rids/20060922/i/r83709231.jpg?x=47&y=130&sig=df38yh9mhxp9zblqrj3smw--" align="left" height="130" width="47" alt="an aerial view shows the crash site of a transrapid high-speed train in the northern german region of emsland near the town of lathen, september 22, 2006. reuters/bodo marks" border="0" /></a>reuters - a high tech german train \
carrying 29 people smashed into a maintenance vehicle on a \
trial run in northern germany on friday, killing at least one \
and trapping 10 more in the wreckage, a local official said.</p><br clear="all"/>
alt captions->
 an aerial view shows the crash site of a transrapid high speed train in the northern german region of emsland near the town of lathen  september           reuters bodo marks
applying removing url function
     reuters - a high tech german train \

In [ ]:

# teest on extracting url informatioon

test = df.iloc[4]['article']
print(test)
test = clean_text(test)

lmt = LemmaTokenizer1()
print(lmt(test))

<p><a href="http://us.rd.yahoo.com/dailynews/rss/europe/*http://news.yahoo.com/s/nm/20060922/wl_nm/transport_germany_accident_dc"><img src="http://d.yimg.com/us.yimg.com/p/rids/20060922/i/r83709231.jpg?x=47&y=130&sig=dF38yH9MhxP9zBLqrj3SMw--" align="left" height="130" width="47" alt="An aerial view shows the crash site of a Transrapid high-speed train in the northern German region of Emsland near the town of Lathen, September 22, 2006. REUTERS/Bodo Marks" border="0" /></a>Reuters - A high tech German train \
carrying 29 people smashed into a maintenance vehicle on a \
trial run in northern Germany on Friday, killing at least one \
and trapping 10 more in the wreckage, a local official said.</p><br clear="all"/>
['reuters', 'high', 'tech', 'german', 'train', 'carrying', 'people', 'smashed', 'maintenance', 'vehicle', 'trial', 'run', 'northern', 'germany', 'friday', 'killing', 'least', 'one', 'trapping', 'wreckage', 'local', 'official', 'said', 'yahoo', 'dailynews', 'europe', 'news', 'yah